<a href="https://colab.research.google.com/github/angeruzzi/recommender_system_movielens/blob/main/05_collaborative_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaborative Filtering — MovieLens 100K

## 1. Introdução

Este notebook implementa sistemas de recomendação baseados em **Collaborative Filtering**, utilizando exclusivamente os padrões de interação entre usuários e filmes.

Diferentemente da abordagem Content-Based, os modelos colaborativos não dependem de atributos explícitos dos filmes, como gêneros. As recomendações são produzidas a partir de relações identificadas no comportamento coletivo dos usuários.

Serão avaliadas duas estratégias:

- **User-Based Collaborative Filtering**: identifica usuários com padrões de avaliação semelhantes e utiliza suas preferências para recomendar novos itens;
- **Item-Based Collaborative Filtering**: identifica filmes avaliados de maneira semelhante pelos usuários e utiliza essas relações para produzir recomendações.

Os modelos serão avaliados utilizando o mesmo split temporal e as mesmas métricas Top-K dos experimentos anteriores.

# 2. Preparação

In [21]:
#Imports

!wget -q -O utils.py \
    https://raw.githubusercontent.com/angeruzzi/recommender_system_movielens/main/utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

import utils

#Validação
assert utils.validate_metrics()
assert utils.validate_framework()

print("Framework carregado com sucesso.")

#Configuração experimental:
RELEVANCE_THRESHOLD = 4
TRAIN_RATIO = 0.8
K = 10
RANDOM_STATE = 42
N_NEIGHBORS = 30
MIN_COMMON_ITEMS = 3

#Carregamento
ratings, movies = utils.load_movielens_100k()

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['user_id'].nunique():,}")
print(f"Movies:  {ratings['item_id'].nunique():,}")

#Split
train, test = utils.temporal_split_by_user(
    ratings,
    train_ratio=TRAIN_RATIO
)

evaluation_data = utils.build_evaluation_data(
    train=train,
    test=test,
    relevance_threshold=RELEVANCE_THRESHOLD
)

train_catalog = evaluation_data["train_catalog"]
seen_items = evaluation_data["seen_items"]
ground_truth = evaluation_data["evaluable_ground_truth"]
evaluation_users = evaluation_data["evaluation_users"]

print(f"Train:              {len(train):,}")
print(f"Test:               {len(test):,}")
print(f"Evaluation users:   {len(evaluation_users):,}")
print(f"Train catalog:      {len(train_catalog):,}")

RESULTS_URL = "https://raw.githubusercontent.com/angeruzzi/recommender_system_movielens/main/model_results2.csv"
model_results = pd.read_csv(RESULTS_URL)

assert utils.validate_temporal_split(train, test).all()
print("Protocolo experimental validado.")

Framework carregado com sucesso.
Ratings: 100,000
Users:   943
Movies:  1,682
Train:              79,619
Test:               20,381
Evaluation users:   907
Train catalog:      1,611
Protocolo experimental validado.


## 3. Matriz Usuário × Item

Os modelos de Collaborative Filtering serão construídos a partir da matriz usuário-item formada apenas pelas interações do conjunto de treino.

As linhas representam usuários, as colunas representam filmes e as células contêm os ratings observados.

In [3]:
user_item_matrix = train.pivot(
    index="user_id",
    columns="item_id",
    values="rating"
)

user_item_matrix.shape

(943, 1611)

### 3.1 Centralização dos Ratings

Usuários podem apresentar diferentes padrões na utilização da escala de avaliações.

Para reduzir esse efeito, os ratings serão centralizados pela média de cada usuário:

\[
r'_{ui}=r_{ui}-\bar r_u
\]

Assim, os valores passam a representar desvios em relação ao comportamento habitual do próprio usuário.

In [6]:
user_means = user_item_matrix.mean(axis=1)

centered_matrix = user_item_matrix.sub(
    user_means,
    axis=0
)

#Para calcular similaridade por cosseno, podemos utilizar zero apenas como representação computacional da ausência de informação após centralização
centered_filled = centered_matrix.fillna(0)

centered_filled.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1660,1662,1663,1664,1670,1672,1673,1675,1676,1681
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.405530,-0.594470,0.0,-0.59447,0.0,0.0,0.40553,-2.59447,0.0,-0.594470,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.285714,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,-1.714286,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.964286,-0.035714,0.0,0.00000,0.0,0.0,0.00000,0.00000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. User-Based Collaborative Filtering

O User-Based Collaborative Filtering procura usuários com padrões de preferência semelhantes.

Para um usuário \(u\), o modelo identifica os usuários mais similares e utiliza suas avaliações para estimar a preferência de \(u\) por itens ainda não avaliados.

### 4.1 Similaridade entre Usuários

A similaridade entre dois usuários será calculada considerando exclusivamente os itens avaliados por ambos (*co-rated items*).

Essa restrição evita interpretar a ausência de uma avaliação como um valor numérico e torna a comparação baseada apenas em evidências efetivamente compartilhadas pelos usuários.

Será utilizada a correlação de Pearson, que mede a similaridade entre os padrões relativos de avaliação dos dois usuários.

Também será exigido um número mínimo de itens coavaliados para reduzir similaridades pouco confiáveis.

In [29]:
def compute_user_similarity(
    user_item_matrix,
    min_common_items=3
):
    user_ids = user_item_matrix.index
    n_users = len(user_ids)

    similarity = np.zeros(
        (n_users, n_users),
        dtype=float
    )

    overlap = np.zeros(
        (n_users, n_users),
        dtype=int
    )

    for i in range(n_users):
        ratings_i = user_item_matrix.iloc[i]

        for j in range(i + 1, n_users):
            ratings_j = user_item_matrix.iloc[j]

            common = (
                ratings_i.notna()
                & ratings_j.notna()
            )

            n_common = common.sum()

            overlap[i, j] = n_common
            overlap[j, i] = n_common

            if n_common < min_common_items:
                continue

            x = ratings_i[common].values
            y = ratings_j[common].values

            # Pearson
            if np.std(x) == 0 or np.std(y) == 0:
                sim = 0.0
            else:
                sim = np.corrcoef(x, y)[0, 1]

            similarity[i, j] = sim
            similarity[j, i] = sim

    similarity_df = pd.DataFrame(
        similarity,
        index=user_ids,
        columns=user_ids
    )

    overlap_df = pd.DataFrame(
        overlap,
        index=user_ids,
        columns=user_ids
    )

    return similarity_df, overlap_df

In [30]:
# user_similarity_values = cosine_similarity(
#     centered_filled.values
# )

# user_similarity = pd.DataFrame(
#     user_similarity_values,
#     index=centered_filled.index,
#     columns=centered_filled.index
# )

# np.fill_diagonal(
#     user_similarity.values,
#     0
# )

# user_similarity.head()

In [8]:
user_similarity_pearson, user_overlap_pearson = (
    compute_user_similarity(
        user_item_matrix,
        min_common_items=MIN_COMMON_ITEMS
    )
)

### 4.2 Sobreposição entre Usuários

A similaridade entre dois usuários só é calculada quando existe uma quantidade mínima de itens avaliados por ambos.

A função utilizada na etapa anterior também retorna uma matriz de sobreposição, na qual cada célula representa o número de itens coavaliados por um par de usuários.

Pares com menos de `MIN_COMMON_ITEMS` interações em comum recebem similaridade igual a zero e não participam da formação da vizinhança.

In [46]:
user_overlap_pearson.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,0,12,3,2,55,61,108,26,3,58,...,60,8,29,9,23,8,34,5,18,60
2,12,0,7,5,5,24,13,6,4,13,...,11,10,24,13,20,10,11,5,6,7
3,3,7,0,8,0,6,5,5,1,5,...,2,1,12,6,7,2,11,3,7,0
4,2,5,8,0,2,4,5,4,1,2,...,3,1,6,5,5,1,6,2,6,3
5,55,5,0,2,0,24,76,15,3,26,...,42,6,10,3,11,5,18,3,11,40


In [47]:
overlap_values = user_overlap_pearson.values

positive_overlaps = overlap_values[
    overlap_values > 0
]

print(f"Média de itens em comum:   {positive_overlaps.mean():.2f}")
print(f"Mediana de itens em comum: {np.median(positive_overlaps):.2f}")
print(f"Máximo de itens em comum:  {positive_overlaps.max()}")

Média de itens em comum:   14.21
Mediana de itens em comum: 7.00
Máximo de itens em comum:  270


In [49]:
invalid_pairs = (
    (user_overlap_pearson < MIN_COMMON_ITEMS)
    & (user_similarity_pearson != 0)
)

print(
    "Pares abaixo do mínimo com similaridade diferente de zero:",
    invalid_pairs.sum().sum()
)

Pares abaixo do mínimo com similaridade diferente de zero: 0


### 4.3 Seleção dos Vizinhos

Para cada usuário, serão selecionados os usuários com maior similaridade de Pearson entre aqueles que possuem quantidade mínima de itens coavaliados.

Como pares com sobreposição insuficiente já receberam similaridade igual a zero na etapa anterior, eles são automaticamente excluídos da vizinhança.

Os vizinhos são ordenados da maior para a menor similaridade e os `N_NEIGHBORS` mais próximos são utilizados na predição.

In [50]:
def get_user_neighbors(
    user_id,
    similarity_matrix,
    n_neighbors=30
):
    similarities = (
        similarity_matrix
        .loc[user_id]
        .drop(index=user_id, errors="ignore")
    )

    # Remove pares sem similaridade válida
    similarities = similarities[
        similarities != 0
    ]

    # Mais similares primeiro
    similarities = similarities.sort_values(
        ascending=False
    )

    return similarities.head(n_neighbors)

In [51]:
user_id = evaluation_users[0]

neighbors = get_user_neighbors(
    user_id=user_id,
    similarity_matrix=user_similarity_pearson,
    n_neighbors=N_NEIGHBORS
)

neighbors.head(10)

,1
user_id,
139,1.000000
724,0.980196
570,0.970725
803,0.965824
905,0.963087
520,0.960769
772,0.944911
926,0.944911
572,0.944911


In [52]:
print(f"Usuário analisado: {user_id}")
print(f"Número de vizinhos encontrados: {len(neighbors)}")
print(f"Maior similaridade: {neighbors.max():.4f}")
print(f"Menor similaridade entre os selecionados: {neighbors.min():.4f}")

Usuário analisado: 1
Número de vizinhos encontrados: 30
Maior similaridade: 1.0000
Menor similaridade entre os selecionados: 0.7778


### 4.4 Predição de Scores

Para cada item candidato, o score será estimado a partir das avaliações dos usuários vizinhos que interagiram com esse item.

Como os ratings dos vizinhos foram centralizados em relação às suas próprias médias, cada avaliação representa um desvio em relação ao comportamento habitual daquele usuário.

Esses desvios são ponderados pela similaridade entre o vizinho e o usuário alvo. A média ponderada resultante é então adicionada à média histórica do usuário alvo, produzindo um rating estimado para o item.

In [54]:
def predict_user_based_scores(
    user_id,
    centered_matrix,
    user_means,
    similarity_matrix,
    train_catalog,
    seen_items,
    n_neighbors=30
):

    # Seleciona os vizinhos mais similares
    neighbors = get_user_neighbors(
        user_id=user_id,
        similarity_matrix=similarity_matrix,
        n_neighbors=n_neighbors
    )

    if neighbors.empty:
        return pd.Series(dtype=float)

    # Itens do catálogo de treino ainda não vistos pelo usuário
    candidates = utils.get_candidate_items(
        user_id=user_id,
        train_catalog=train_catalog,
        seen_items=seen_items
    )

    scores = {}

    # Calcula um score para cada item candidato
    for item_id in candidates:

        if item_id not in centered_matrix.columns:
            continue

        # Ratings centralizados dos vizinhos para o item
        neighbor_ratings = centered_matrix.loc[
            neighbors.index,
            item_id
        ]

        # Mantém apenas vizinhos que avaliaram o item
        available = neighbor_ratings.notna()

        if not available.any():
            continue

        item_neighbors = neighbor_ratings[available]

        # Similaridades dos vizinhos que avaliaram o item
        similarities = neighbors.loc[
            item_neighbors.index
        ]

        denominator = np.abs(similarities).sum()

        if denominator == 0:
            continue

        # Média ponderada dos desvios dos ratings
        weighted_deviation = (
            similarities * item_neighbors
        ).sum() / denominator

        # Retorna o score para a escala original do usuário
        predicted_rating = (
            user_means.loc[user_id]
            + weighted_deviation
        )

        scores[item_id] = predicted_rating

    return pd.Series(
        scores,
        dtype=float
    ).sort_values(ascending=False)

In [55]:
# Teste

user_id = evaluation_users[0]

user_scores = predict_user_based_scores(
    user_id=user_id,
    centered_matrix=centered_matrix,
    user_means=user_means,
    similarity_matrix=user_similarity_pearson,
    train_catalog=train_catalog,
    seen_items=seen_items,
    n_neighbors=N_NEIGHBORS
)

user_scores.head(10)

,0
287,6.049016
316,5.488588
902,5.488588
475,5.125720
298,5.125720
477,5.112989
297,4.829983
334,4.790825
285,4.746262
272,4.725401


### 4.5 Recomendações Top-K

Os itens candidatos serão ordenados pelo rating estimado. Os \(K\) itens com maior score formarão a lista final de recomendações do User-Based Collaborative Filtering.

In [56]:
user_cf_recommendations = {}

for user_id in evaluation_users:

    scores = predict_user_based_scores(
        user_id=user_id,
        centered_matrix=centered_matrix,
        user_means=user_means,
        similarity_matrix=user_similarity_pearson,
        train_catalog=train_catalog,
        seen_items=seen_items,
        n_neighbors=N_NEIGHBORS
    )

    user_cf_recommendations[user_id] = (
        scores
        .head(K)
        .index
        .tolist()
    )

In [58]:
#Validação do tamanho das listas
recommendation_lengths = pd.Series({
    user_id: len(items)
    for user_id, items in user_cf_recommendations.items()
})

recommendation_lengths.describe()

,0
count,907.0
mean,10.0
std,0.0
min,10.0
25%,10.0
50%,10.0
75%,10.0
max,10.0


### 4.6 Avaliação do User-Based CF

In [59]:
user_cf_results, user_cf_summary = (
    utils.evaluate_recommendations(
        recommendations=user_cf_recommendations,
        ground_truth=ground_truth,
        k=K
    )
)

user_cf_summary

{'Precision@10': np.float64(0.013671444321940463),
 'Recall@10': np.float64(0.01193955057281341),
 'NDCG@10': np.float64(0.015679018530965115),
 'n_users': 907}

In [60]:
comparison = utils.add_model_result(
    results_df=model_results,
    model_name="User-Based CF",
    summary=user_cf_summary,
    k=K
)

comparison

,model,Precision@10,Recall@10,NDCG@10,n_users
0,Random,0.007387,0.006949,0.008209,907
1,Popularity,0.077508,0.078646,0.098883,907
2,Content-Based,0.018964,0.018318,0.025018,907
3,User-Based CF,0.013671,0.011940,0.015679,907


### 4.7 Análise dos Resultados

O User-Based Collaborative Filtering apresentou desempenho superior ao Random Baseline nas três métricas avaliadas, indicando que a similaridade entre usuários contém informação útil para a geração das recomendações.

Entretanto, o modelo apresentou desempenho inferior ao Content-Based e, principalmente, ao Popularity Baseline.

A diferença em relação ao Popularity sugere que a popularidade global dos itens representa um sinal particularmente forte neste conjunto de dados e protocolo experimental.

O resultado também evidencia algumas limitações do User-Based CF baseado em vizinhança. A matriz usuário-item é esparsa, a similaridade entre usuários depende da existência de itens coavaliados e as estimativas para cada item são produzidas a partir de um subconjunto variável dos vizinhos disponíveis.

Apesar disso, o resultado representa uma melhora em relação à primeira implementação baseada em similaridade de cosseno com preenchimento dos valores ausentes. A utilização da correlação de Pearson calculada exclusivamente sobre itens coavaliados produziu uma vizinhança mais adequada ao problema de ratings explícitos.

A próxima etapa será avaliar o Item-Based Collaborative Filtering, no qual as relações de similaridade serão estabelecidas entre filmes em vez de usuários.